# 08_FIXED_Spatial_Downscaling_No_Artifacts

This notebook rebuilds the spatial downscaling stage using safer raster handling.

Main fixes:
- one trusted reference raster/grid;
- `rasterio.warp.reproject()` instead of manual SciPy pixel-coordinate sampling;
- preserve source NoData and missing strips;
- no nearest filling of large internal gaps;
- bilinear for continuous variables, nearest for categorical variables;
- prediction only where all required predictors are valid;
- Khulna boundary mask;
- monthly + annual GeoTIFF export;
- predictor and output seam diagnostics.

In [1]:
# Optional installs if needed:
# %pip install rasterio geopandas pandas numpy matplotlib scikit-learn joblib
# %pip install xgboost lightgbm catboost

In [2]:
from pathlib import Path
from collections import OrderedDict
import re, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import geometry_mask
import geopandas as gpd
import joblib

from sklearn.ensemble import RandomForestRegressor

RANDOM_STATE = 42
NODATA = -9999.0
TEST_YEAR = 2022
MIN_VALID_COVERAGE = 0.90
SEAM_Z_THRESHOLD = 8.0

print("Rasterio:", rasterio.__version__)

Rasterio: 1.4.4


## 1. Project paths and settings

If auto-detection fails, edit `PROJECT_ROOT`, `REFERENCE_RASTER`, and `BOUNDARY_FILE`.

In [3]:
def find_project_root(start=None):
    cur = (Path(start) if start else Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p/"data").exists():
            return p
    return cur

PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT/"data"
RAW_DIR = DATA_DIR/"raw"
PROCESSED_DIR = DATA_DIR/"processed"

ALIGNED_ROOT = PROCESSED_DIR/"aligned_rasters"
PRECIP_ROOT = ALIGNED_ROOT/"precipitation"
PRED_ROOT = ALIGNED_ROOT/"predictors"
RAW_PRED_ROOT = RAW_DIR/"predictors"

OUT_ROOT = PROJECT_ROOT/"outputs"/"fixed_spatial_downscaling_no_artifacts"
TABLE_DIR = OUT_ROOT/"tables"
QA_DIR = OUT_ROOT/"quality_control"
FIG_DIR = OUT_ROOT/"figures"
MONTHLY_DIR = OUT_ROOT/"monthly_masked_tif"
ANNUAL_DIR = OUT_ROOT/"annual_masked_tif"
MODEL_DIR = PROJECT_ROOT/"models"/"paper_style_proj_free_final"

for p in [OUT_ROOT,TABLE_DIR,QA_DIR,FIG_DIR,MONTHLY_DIR,ANNUAL_DIR,MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

REFERENCE_RASTER = None
BOUNDARY_FILE = None

CATEGORICAL_DATASETS = {"LULC","LANDCOVER","LAND_USE","LAND_USE_LAND_COVER"}
LAND_FEATURES = ["DEM","NDVI","LST_DAY","DIST_SEA"]

COMBINATIONS = OrderedDict({
    "Comb1": ["CCS","CDR","CHIRPS","ERA5","GSMAP_GAUGE","GSMAP_MVK","IMERG","PDIR","PERSIANN"],
    "Comb1_land": ["CCS","CDR","CHIRPS","ERA5","GSMAP_GAUGE","GSMAP_MVK","IMERG","PDIR","PERSIANN"] + LAND_FEATURES,
    "Comb2": ["CHIRPS","CDR","PERSIANN","ERA5"],
    "Comb2_land": ["CHIRPS","CDR","PERSIANN","ERA5"] + LAND_FEATURES,
    "Comb3": ["CCS","PDIR"],
    "Comb3_land": ["CCS","PDIR"] + LAND_FEATURES,
})

for _p in ["CCS","CDR","CHIRPS","ERA5","GSMAP_GAUGE","GSMAP_MVK","IMERG","PDIR","PERSIANN"]:
    COMBINATIONS[f"{_p}_land"] = [_p] + LAND_FEATURES

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT:", OUT_ROOT)

PROJECT_ROOT: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh
OUTPUT: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\outputs\fixed_spatial_downscaling_no_artifacts


## 2. Dataset indexing

In [4]:
def norm_name(x):
    return re.sub(r"[^A-Z0-9]+","_",str(x).upper()).strip("_")

ALIASES = {
    "PERSIANN_CDR":"CDR",
    "PERSIANNCDR":"CDR",
    "PERSIANN_CCS":"CCS",
    "PDIR_NOW":"PDIR",
    "GSMAP_GAUGE_V7":"GSMAP_GAUGE",
    "GSMAP_MVK_V7":"GSMAP_MVK",
    "ERA5_TIFF":"ERA5",
    "CHIRPS_TIFF_2017_2022":"CHIRPS",
    "IMERG_MONTHLY":"IMERG",
    "LST_DAYTIME":"LST_DAY",
    "DISTANCE_TO_SEA":"DIST_SEA",
    "DISTANCE_SEA":"DIST_SEA",
    "ELEVATION":"DEM",
}

def canonical(x):
    n=norm_name(x)
    return ALIASES.get(n,n)

def all_tifs(folder):
    if folder is None or not Path(folder).exists():
        return []
    z=[]
    for pat in ["*.tif","*.tiff","*.TIF","*.TIFF"]:
        z.extend(Path(folder).rglob(pat))
    return sorted(set(z))

def extract_ym(path):
    s=Path(path).stem
    m=re.search(r"(?<!\d)(20\d{2})[^0-9]+(0?[1-9]|1[0-2])(?!\d)",s)
    if m:
        return int(m.group(1)),int(m.group(2))
    m=re.search(r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)",s)
    if m:
        return int(m.group(1)),int(m.group(2))
    return None

def folder_map(root):
    d={}
    if root.exists():
        for p in root.iterdir():
            if p.is_dir():
                d[canonical(p.name)] = p
    return d

def month_index(folder):
    d={}
    for p in all_tifs(folder):
        ym=extract_ym(p)
        if ym:
            d[ym]=p
    return d

precip_folders=folder_map(PRECIP_ROOT)
predictor_folders=folder_map(PRED_ROOT)
raw_predictor_folders=folder_map(RAW_PRED_ROOT)

MONTHLY_INDEX={}
for name,folder in {**precip_folders,**predictor_folders}.items():
    MONTHLY_INDEX[name]=month_index(folder)

STATIC_FILES={}
for name,folder in {**predictor_folders,**raw_predictor_folders}.items():
    c=[p for p in all_tifs(folder) if extract_ym(p) is None]
    if c:
        STATIC_FILES[name]=c[0]

rows=[]
for name in sorted(set(MONTHLY_INDEX)|set(STATIC_FILES)):
    idx=MONTHLY_INDEX.get(name,{})
    rows.append({
        "dataset":name,
        "monthly_count":len(idx),
        "has_all_2022": all((TEST_YEAR,m) in idx for m in range(1,13)) if idx else False,
        "static_file":str(STATIC_FILES.get(name,""))
    })

inventory=pd.DataFrame(rows)
display(inventory)
inventory.to_csv(TABLE_DIR/"dataset_inventory_fixed.csv",index=False)

,dataset,monthly_count,has_all_2022,static_file
0,ASPECT,0,False,E:\Geospatial\Precipitation-Downscaling-Khulna...
1,CCS,72,True,
2,CDR,72,True,
3,CHIRPS,72,True,
4,DEM,0,False,E:\Geospatial\Precipitation-Downscaling-Khulna...
5,DIST_SEA,0,False,E:\Geospatial\Precipitation-Downscaling-Khulna...
6,ERA5,72,True,
7,GSMAP_GAUGE,72,True,
8,GSMAP_MVK,72,True,
9,IMERG,72,True,


## 3. Boundary detection

In [5]:
def auto_find_boundary():
    cand=[]
    for root in [RAW_DIR,PROCESSED_DIR,PROJECT_ROOT]:
        if not root.exists():
            continue
        for ext in ["*.shp","*.gpkg","*.geojson"]:
            cand += list(root.rglob(ext))
    scored=[]
    for p in cand:
        s=p.name.lower()
        score=int("khulna" in s)*5 + int("district" in s)*3 + int("bound" in s)*2
        scored.append((score,p))
    scored.sort(key=lambda x:(-x[0],len(str(x[1]))))
    return scored[0][1] if scored else None

if BOUNDARY_FILE is None:
    BOUNDARY_FILE=auto_find_boundary()

if BOUNDARY_FILE is None:
    raise FileNotFoundError("Boundary not found. Set BOUNDARY_FILE manually.")

boundary=gpd.read_file(BOUNDARY_FILE)
boundary=boundary[boundary.geometry.notna()].copy().dissolve()

print("Boundary:",BOUNDARY_FILE)
print("Boundary CRS:",boundary.crs)

Boundary: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\boundary\Khulna.shp
Boundary CRS: EPSG:4326


## 4. Safe reference grid selection

Automatic selection prefers a raster that covers the full Khulna bounding box and has the finest resolution.
You can override it by setting `REFERENCE_RASTER` manually.

In [6]:
def raster_info(path):
    with rasterio.open(path) as src:
        return {
            "path":Path(path),
            "crs":src.crs,
            "transform":src.transform,
            "width":src.width,
            "height":src.height,
            "bounds":src.bounds,
            "res":(abs(src.transform.a),abs(src.transform.e)),
            "nodata":src.nodata,
        }

def coverage_of_boundary(path):
    with rasterio.open(path) as src:
        b=boundary.to_crs(src.crs)
        bb=b.total_bounds
        rb=src.bounds
        ix0=max(rb.left,bb[0]); iy0=max(rb.bottom,bb[1])
        ix1=min(rb.right,bb[2]); iy1=min(rb.top,bb[3])
        inter=max(0,ix1-ix0)*max(0,iy1-iy0)
        area=max(1e-12,(bb[2]-bb[0])*(bb[3]-bb[1]))
        return inter/area

def choose_reference():
    candidates=[]
    for key in ["DEM","NDVI","LST_DAY","DIST_SEA"]:
        folder=predictor_folders.get(key) or raw_predictor_folders.get(key)
        for p in all_tifs(folder)[:100]:
            try:
                info=raster_info(p)
                cov=coverage_of_boundary(p)
                pix=info["res"][0]*info["res"][1]
                candidates.append((cov,pix,p))
            except Exception:
                pass
    if not candidates:
        raise FileNotFoundError("No reference candidates found.")
    good=[x for x in candidates if x[0]>=0.995]
    use=good if good else candidates
    use.sort(key=lambda x:(-x[0],x[1]))
    return use[0][2]

if REFERENCE_RASTER is None:
    REFERENCE_RASTER=choose_reference()

REF=raster_info(REFERENCE_RASTER)
cov=coverage_of_boundary(REFERENCE_RASTER)

print("REFERENCE:",REFERENCE_RASTER)
print("CRS:",REF["crs"])
print("Shape:",REF["height"],REF["width"])
print("Resolution:",REF["res"])
print("Boundary bbox coverage:",f"{cov*100:.2f}%")

if cov < 0.995:
    raise ValueError("Reference raster does not fully cover Khulna. Set REFERENCE_RASTER manually.")

boundary_ref=boundary.to_crs(REF["crs"])
khulna_mask=geometry_mask(
    boundary_ref.geometry,
    out_shape=(REF["height"],REF["width"]),
    transform=REF["transform"],
    invert=True
)

REFERENCE: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\aligned_rasters\predictors\DEM\Khulna_SRTM_DEM.tif
CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Shape: 151 59
Resolution: (0.008983152841195215, 0.008983152841195215)
Boundary bbox coverage: 100.00%


## 5. Safe reprojection/resampling

No nearest filling of internal missing areas is used.
A separate source-validity mask is reprojected so missing strips remain missing.

In [7]:
def dataset_resampling(dataset):
    return Resampling.nearest if canonical(dataset) in CATEGORICAL_DATASETS else Resampling.bilinear

def source_valid_mask(src,arr):
    valid=np.isfinite(arr)
    if src.nodata is not None and np.isfinite(src.nodata):
        valid &= arr != src.nodata
    try:
        valid &= src.read_masks(1)>0
    except Exception:
        pass
    return valid

def reproject_to_reference(path,dataset):
    with rasterio.open(path) as src:
        arr=src.read(1).astype("float32")
        valid_src=source_valid_mask(src,arr)

        src_data=arr.copy()
        src_data[~valid_src]=NODATA

        dst=np.full((REF["height"],REF["width"]),np.nan,dtype="float32")

        reproject(
            source=src_data,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=NODATA,
            dst_transform=REF["transform"],
            dst_crs=REF["crs"],
            dst_nodata=np.nan,
            resampling=dataset_resampling(dataset),
            init_dest_nodata=True
        )

        valid_dst=np.zeros((REF["height"],REF["width"]),dtype="uint8")
        reproject(
            source=valid_src.astype("uint8"),
            destination=valid_dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=0,
            dst_transform=REF["transform"],
            dst_crs=REF["crs"],
            dst_nodata=0,
            resampling=Resampling.nearest,
            init_dest_nodata=True
        )

        dst[valid_dst==0]=np.nan
        dst[~khulna_mask]=np.nan
        return dst

def scale_feature(dataset,arr):
    a=arr.astype("float32").copy()
    vals=a[np.isfinite(a)]
    if vals.size==0:
        return a
    if canonical(dataset)=="NDVI":
        if np.nanpercentile(np.abs(vals),95)>2:
            a *= 0.0001
    return a

## 6. Seam and valid-coverage diagnostics

In [8]:
def robust_jump_profile(v):
    d=np.abs(np.diff(v))
    f=d[np.isfinite(d)]
    if len(f)==0:
        return d,np.full_like(d,np.nan)
    med=np.nanmedian(f)
    mad=np.nanmedian(np.abs(f-med))
    scale=max(1e-12,1.4826*mad)
    return d,(d-med)/scale

def seam_metrics(arr):
    row=np.nanmean(arr,axis=1)
    col=np.nanmean(arr,axis=0)
    _,rz=robust_jump_profile(row)
    _,cz=robust_jump_profile(col)
    return {
        "max_row_jump_z":float(np.nanmax(rz)) if np.isfinite(rz).any() else np.nan,
        "max_col_jump_z":float(np.nanmax(cz)) if np.isfinite(cz).any() else np.nan,
        "row_index":int(np.nanargmax(rz)) if np.isfinite(rz).any() else -1,
        "col_index":int(np.nanargmax(cz)) if np.isfinite(cz).any() else -1,
    }

def valid_coverage(arr):
    return float(np.isfinite(arr[khulna_mask]).sum()/max(1,int(khulna_mask.sum())))

def quicklook(arr,title,outfile):
    plt.figure(figsize=(7,9))
    vals=arr[np.isfinite(arr)]
    if vals.size:
        vmin,vmax=np.nanpercentile(vals,[2,98])
        plt.imshow(arr,vmin=vmin,vmax=vmax)
        plt.colorbar(label="Value")
    else:
        plt.imshow(arr)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(outfile,dpi=200)
    plt.close()

def get_feature_path(dataset,year,month):
    d=canonical(dataset)
    if (year,month) in MONTHLY_INDEX.get(d,{}):
        return MONTHLY_INDEX[d][(year,month)]
    return STATIC_FILES.get(d)

needed=sorted(set(f for fs in COMBINATIONS.values() for f in fs))
qa_rows=[]

for feat in needed:
    for month in range(1,13):
        p=get_feature_path(feat,TEST_YEAR,month)
        if p is None:
            qa_rows.append({"dataset":feat,"month":month,"status":"MISSING","valid_coverage":0})
            continue
        try:
            a=scale_feature(feat,reproject_to_reference(p,feat))
            cov=valid_coverage(a)
            sm=seam_metrics(a)
            qa_rows.append({
                "dataset":feat,"month":month,"status":"OK" if cov>=MIN_VALID_COVERAGE else "LOW_COVERAGE",
                "valid_coverage":cov,"path":str(p),**sm
            })
        except Exception as e:
            qa_rows.append({"dataset":feat,"month":month,"status":"ERROR","path":str(p),"detail":repr(e)})

qa=pd.DataFrame(qa_rows)
qa.to_csv(QA_DIR/"predictor_QA_2022.csv",index=False)

problems=qa[
    (qa["status"]!="OK") |
    (qa["max_row_jump_z"].fillna(-999)>=SEAM_Z_THRESHOLD) |
    (qa["max_col_jump_z"].fillna(-999)>=SEAM_Z_THRESHOLD)
].copy()

problems.to_csv(QA_DIR/"predictor_QA_2022_PROBLEMS.csv",index=False)
print("Potential predictor problems:",len(problems))
display(problems.head(100))

Potential predictor problems: 151


,dataset,month,status,valid_coverage,path,max_row_jump_z,max_col_jump_z,row_index,col_index
0,CCS,1,OK,0.948729,E:\Geospatial\Precipitation-Downscaling-Khulna...,17.863531,18.047516,32,43
2,CCS,3,OK,0.948729,E:\Geospatial\Precipitation-Downscaling-Khulna...,7.566356,9.198634,18,5
3,CCS,4,OK,0.948729,E:\Geospatial\Precipitation-Downscaling-Khulna...,18.939638,21.792833,120,9
4,CCS,5,OK,0.948729,E:\Geospatial\Precipitation-Downscaling-Khulna...,5.691766,39.415043,18,43
5,CCS,6,OK,0.948729,E:\Geospatial\Precipitation-Downscaling-Khulna...,8.956389,13.579421,3,1
...,...,...,...,...,...,...,...,...,...
99,IMERG,4,OK,0.901178,E:\Geospatial\Precipitation-Downscaling-Khulna...,15.009605,28.072617,140,43
100,IMERG,5,OK,0.901178,E:\Geospatial\Precipitation-Downscaling-Khulna...,15.054503,19.547255,148,43
101,IMERG,6,OK,0.901178,E:\Geospatial\Precipitation-Downscaling-Khulna...,9.637835,9.343937,17,43
102,IMERG,7,OK,0.901178,E:\Geospatial\Precipitation-Downscaling-Khulna...,13.570605,32.523941,148,35


## 7. Build feature stacks with strict valid-pixel intersection

In [9]:
FEATURE_CACHE={}

def get_feature_array(dataset,year,month):
    key=(canonical(dataset),year,month)
    if key in FEATURE_CACHE:
        return FEATURE_CACHE[key]
    p=get_feature_path(dataset,year,month)
    if p is None:
        raise FileNotFoundError(f"No raster: {dataset} {year}-{month:02d}")
    a=scale_feature(dataset,reproject_to_reference(p,dataset))
    FEATURE_CACHE[key]=a
    return a

def build_stack(features,year,month):
    arrs=[]
    mask=khulna_mask.copy()
    for f in features:
        a=get_feature_array(f,year,month)
        arrs.append(a)
        mask &= np.isfinite(a)
    X=np.stack(arrs,axis=-1)
    return X,mask

X0,M0=build_stack(COMBINATIONS["Comb2_land"],TEST_YEAR,1)
print("Comb2_land Jan valid coverage:",f"{M0.sum()/khulna_mask.sum()*100:.2f}%")

Comb2_land Jan valid coverage: 41.08%


## 8. Load original saved models

If no saved models exist, the notebook will skip those model outputs by default.
This avoids silently changing your model methodology.

In [10]:
MODEL_NAMES=["Random_Forest","XGBoost","LightGBM","CatBoost"]

def model_files():
    z=[]
    if MODEL_DIR.exists():
        for ext in ["*.joblib","*.pkl","*.pickle"]:
            z += list(MODEL_DIR.rglob(ext))
    return sorted(set(z))

ALL_MODEL_FILES=model_files()
print("Saved models found:",len(ALL_MODEL_FILES))
for p in ALL_MODEL_FILES[:30]:
    print(" ",p.name)

def find_saved_model(combo,model_name):
    c=norm_name(combo)
    m=norm_name(model_name)
    hits=[]
    for p in ALL_MODEL_FILES:
        s=norm_name(p.stem)
        ok_combo=c in s
        ok_model=m in s
        if model_name=="Random_Forest":
            ok_model=ok_model or "RANDOM_FOREST" in s or s.endswith("_RF")
        if ok_combo and ok_model:
            hits.append(p)
    return hits[0] if hits else None

def load_model(combo,model_name):
    p=find_saved_model(combo,model_name)
    if p is None:
        return None,None
    try:
        return joblib.load(p),p
    except Exception:
        return None,p

Saved models found: 60
  CCS_land__CatBoost.joblib
  CCS_land__LightGBM.joblib
  CCS_land__Random_Forest.joblib
  CCS_land__XGBoost.joblib
  CDR_land__CatBoost.joblib
  CDR_land__LightGBM.joblib
  CDR_land__Random_Forest.joblib
  CDR_land__XGBoost.joblib
  CHIRPS_land__CatBoost.joblib
  CHIRPS_land__LightGBM.joblib
  CHIRPS_land__Random_Forest.joblib
  CHIRPS_land__XGBoost.joblib
  Comb1__CatBoost.joblib
  Comb1__LightGBM.joblib
  Comb1__Random_Forest.joblib
  Comb1__XGBoost.joblib
  Comb1_land__CatBoost.joblib
  Comb1_land__LightGBM.joblib
  Comb1_land__Random_Forest.joblib
  Comb1_land__XGBoost.joblib
  Comb2__CatBoost.joblib
  Comb2__LightGBM.joblib
  Comb2__Random_Forest.joblib
  Comb2__XGBoost.joblib
  Comb2_land__CatBoost.joblib
  Comb2_land__LightGBM.joblib
  Comb2_land__Random_Forest.joblib
  Comb2_land__XGBoost.joblib
  Comb3__CatBoost.joblib
  Comb3__LightGBM.joblib


## 9. Optional fallback training

Keep `TRAIN_IF_MODEL_MISSING=False` if you want to preserve the original trained models only.
Turn it on only when you have the exact station training table and accept retraining.

In [11]:
TRAIN_IF_MODEL_MISSING=False
TRAINING_TABLE=None
TARGET_COLUMN="GAUGE"

def create_model(name):
    if name=="Random_Forest":
        return RandomForestRegressor(n_estimators=500,min_samples_leaf=2,random_state=42,n_jobs=-1)
    if name=="XGBoost":
        from xgboost import XGBRegressor
        return XGBRegressor(n_estimators=500,max_depth=4,learning_rate=0.03,subsample=0.8,colsample_bytree=0.8,random_state=42,n_jobs=-1)
    if name=="LightGBM":
        from lightgbm import LGBMRegressor
        return LGBMRegressor(n_estimators=500,learning_rate=0.03,num_leaves=31,random_state=42,verbosity=-1)
    if name=="CatBoost":
        from catboost import CatBoostRegressor
        return CatBoostRegressor(iterations=500,depth=6,learning_rate=0.03,random_seed=42,verbose=False)
    raise ValueError(name)

def train_fallback_model(combo,model_name):
    if not TRAIN_IF_MODEL_MISSING:
        return None
    if TRAINING_TABLE is None or not Path(TRAINING_TABLE).exists():
        raise FileNotFoundError("Set TRAINING_TABLE.")
    df=pd.read_csv(TRAINING_TABLE)
    feats=COMBINATIONS[combo]
    d=df[feats+[TARGET_COLUMN]].replace([np.inf,-np.inf],np.nan).dropna()
    model=create_model(model_name)
    model.fit(d[feats],d[TARGET_COLUMN])
    return model

## 10. GeoTIFF writer

In [12]:
def write_tif(path,arr):
    path=Path(path)
    out=np.asarray(arr,dtype="float32").copy()
    out[~np.isfinite(out)]=NODATA
    profile={
        "driver":"GTiff",
        "height":REF["height"],
        "width":REF["width"],
        "count":1,
        "dtype":"float32",
        "crs":REF["crs"],
        "transform":REF["transform"],
        "nodata":NODATA,
        "compress":"deflate",
        "predictor":3,
        "tiled":True,
        "BIGTIFF":"IF_SAFER"
    }
    with rasterio.open(path,"w",**profile) as dst:
        dst.write(out,1)
    return path

## 11. Generate fixed monthly predictions

In [13]:
RUN_COMBINATIONS=list(COMBINATIONS.keys())
RUN_MODELS=MODEL_NAMES

monthly_paths={}
prediction_qc=[]

for combo in RUN_COMBINATIONS:
    features=COMBINATIONS[combo]

    for model_name in RUN_MODELS:
        model,model_path=load_model(combo,model_name)
        if model is None:
            model=train_fallback_model(combo,model_name)

        if model is None:
            print("SKIP - no model:",combo,model_name)
            prediction_qc.append({"combination":combo,"model":model_name,"status":"NO_MODEL"})
            continue

        print("\\n",combo,"|",model_name)
        paths=[]

        for month in range(1,13):
            try:
                X,mask=build_stack(features,TEST_YEAR,month)
                coverage=mask.sum()/max(1,khulna_mask.sum())

                if coverage < MIN_VALID_COVERAGE:
                    raise RuntimeError(f"Predictor intersection coverage {coverage*100:.2f}% is too low.")

                flat=X[mask,:]
                try:
                    predvals=model.predict(pd.DataFrame(flat,columns=features))
                except Exception:
                    predvals=model.predict(flat)

                predvals=np.asarray(predvals,dtype="float32")
                predvals[predvals<0]=0

                pred=np.full((REF["height"],REF["width"]),np.nan,dtype="float32")
                pred[mask]=predvals
                pred[~khulna_mask]=np.nan

                out=MONTHLY_DIR/f"{TEST_YEAR}_{month:02d}_{combo}_{model_name}_FIXED.tif"
                write_tif(out,pred)
                paths.append(out)

                sm=seam_metrics(pred)
                prediction_qc.append({
                    "combination":combo,"model":model_name,"month":month,"status":"OK",
                    "valid_coverage":coverage,"min":float(np.nanmin(pred)),
                    "max":float(np.nanmax(pred)),"mean":float(np.nanmean(pred)),
                    **sm,"path":str(out),"saved_model":str(model_path or "")
                })

                print(f"{month:02d} OK | {coverage*100:.2f}% | {np.nanmin(pred):.1f}-{np.nanmax(pred):.1f}")

            except Exception as e:
                print(f"{month:02d} ERROR:",e)
                prediction_qc.append({
                    "combination":combo,"model":model_name,"month":month,
                    "status":"ERROR","detail":repr(e)
                })

        monthly_paths[(combo,model_name)]=paths

pred_qc_df=pd.DataFrame(prediction_qc)
pred_qc_df.to_csv(QA_DIR/"monthly_prediction_QA.csv",index=False)
display(pred_qc_df.head(50))

\n Comb1 | Random_Forest
01 ERROR: Predictor intersection coverage 42.46% is too low.
02 ERROR: Predictor intersection coverage 42.46% is too low.
03 ERROR: Predictor intersection coverage 42.46% is too low.
04 ERROR: Predictor intersection coverage 42.46% is too low.
05 ERROR: Predictor intersection coverage 42.46% is too low.
06 ERROR: Predictor intersection coverage 42.46% is too low.
07 ERROR: Predictor intersection coverage 42.46% is too low.
08 ERROR: Predictor intersection coverage 1.34% is too low.
09 ERROR: Predictor intersection coverage 42.46% is too low.
10 ERROR: Predictor intersection coverage 42.46% is too low.
11 ERROR: Predictor intersection coverage 42.46% is too low.
12 ERROR: Predictor intersection coverage 42.46% is too low.
\n Comb1 | XGBoost
01 ERROR: Predictor intersection coverage 42.46% is too low.
02 ERROR: Predictor intersection coverage 42.46% is too low.
03 ERROR: Predictor intersection coverage 42.46% is too low.
04 ERROR: Predictor intersection coverage 

,combination,model,month,status,detail
0,Comb1,Random_Forest,1,ERROR,RuntimeError('Predictor intersection coverage ...
1,Comb1,Random_Forest,2,ERROR,RuntimeError('Predictor intersection coverage ...
2,Comb1,Random_Forest,3,ERROR,RuntimeError('Predictor intersection coverage ...
3,Comb1,Random_Forest,4,ERROR,RuntimeError('Predictor intersection coverage ...
4,Comb1,Random_Forest,5,ERROR,RuntimeError('Predictor intersection coverage ...
5,Comb1,Random_Forest,6,ERROR,RuntimeError('Predictor intersection coverage ...
6,Comb1,Random_Forest,7,ERROR,RuntimeError('Predictor intersection coverage ...
7,Comb1,Random_Forest,8,ERROR,RuntimeError('Predictor intersection coverage ...
8,Comb1,Random_Forest,9,ERROR,RuntimeError('Predictor intersection coverage ...
9,Comb1,Random_Forest,10,ERROR,RuntimeError('Predictor intersection coverage ...


## 12. Annual maps = sum of 12 monthly maps

In [14]:
annual_rows=[]
annual_paths={}

for (combo,model_name),paths in monthly_paths.items():
    if len(paths)!=12:
        annual_rows.append({
            "combination":combo,"model":model_name,
            "status":"SKIPPED","reason":f"{len(paths)}/12 monthly maps"
        })
        continue

    arrs=[]
    for p in paths:
        with rasterio.open(p) as src:
            a=src.read(1).astype("float32")
            a[a==src.nodata]=np.nan
            arrs.append(a)

    stack=np.stack(arrs,axis=0)
    valid=np.all(np.isfinite(stack),axis=0) & khulna_mask

    annual=np.full((REF["height"],REF["width"]),np.nan,dtype="float32")
    annual[valid]=np.sum(stack[:,valid],axis=0)

    out=ANNUAL_DIR/f"annual_{TEST_YEAR}_{combo}_{model_name}_FIXED.tif"
    write_tif(out,annual)
    annual_paths[(combo,model_name)]=out

    sm=seam_metrics(annual)
    annual_rows.append({
        "combination":combo,"model":model_name,"status":"OK",
        "valid_coverage":valid.sum()/khulna_mask.sum(),
        "min":float(np.nanmin(annual)),"max":float(np.nanmax(annual)),
        "mean":float(np.nanmean(annual)),**sm,"path":str(out)
    })

annual_df=pd.DataFrame(annual_rows)
if len(annual_df):
    annual_df["seam_flag"] = (
        annual_df.get("max_row_jump_z",pd.Series(index=annual_df.index,dtype=float)).fillna(-999)>=SEAM_Z_THRESHOLD
    ) | (
        annual_df.get("max_col_jump_z",pd.Series(index=annual_df.index,dtype=float)).fillna(-999)>=SEAM_Z_THRESHOLD
    )

annual_df.to_csv(TABLE_DIR/"annual_output_statistics_FIXED.csv",index=False)
annual_df.to_csv(QA_DIR/"annual_seam_check_FIXED.csv",index=False)
display(annual_df)

,combination,model,status,reason,seam_flag
0,Comb1,Random_Forest,SKIPPED,0/12 monthly maps,False
1,Comb1,XGBoost,SKIPPED,0/12 monthly maps,False
2,Comb1,LightGBM,SKIPPED,0/12 monthly maps,False
3,Comb1,CatBoost,SKIPPED,0/12 monthly maps,False
4,Comb1_land,Random_Forest,SKIPPED,0/12 monthly maps,False
5,Comb1_land,XGBoost,SKIPPED,0/12 monthly maps,False
6,Comb1_land,LightGBM,SKIPPED,0/12 monthly maps,False
7,Comb1_land,CatBoost,SKIPPED,0/12 monthly maps,False
8,Comb2,Random_Forest,SKIPPED,0/12 monthly maps,False
9,Comb2,XGBoost,SKIPPED,0/12 monthly maps,False


## 13. Annual quick-look maps

In [15]:
for (combo,model_name),p in annual_paths.items():
    with rasterio.open(p) as src:
        a=src.read(1).astype("float32")
        a[a==src.nodata]=np.nan
    quicklook(
        a,
        f"Annual precipitation {TEST_YEAR} - {combo} - {model_name} - FIXED",
        FIG_DIR/f"annual_{TEST_YEAR}_{combo}_{model_name}_FIXED.png"
    )

print("Figures saved:",FIG_DIR)

Figures saved: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\outputs\fixed_spatial_downscaling_no_artifacts\figures


## 14. Match prediction seams with predictor seams

In [16]:
match_rows=[]

for _,ar in annual_df.iterrows():
    if ar.get("status")!="OK":
        continue
    feats=COMBINATIONS.get(ar["combination"],[])
    for feat in feats:
        q=qa[qa["dataset"]==feat]
        for _,r in q.iterrows():
            if (
                r.get("max_row_jump_z",np.nan)>=SEAM_Z_THRESHOLD and
                abs(r.get("row_index",-999)-ar.get("row_index",-999))<=3
            ):
                match_rows.append({
                    "combination":ar["combination"],"model":ar["model"],
                    "axis":"row","annual_index":ar.get("row_index"),
                    "predictor":feat,"month":r["month"],
                    "predictor_index":r.get("row_index"),
                    "predictor_jump_z":r.get("max_row_jump_z")
                })
            if (
                r.get("max_col_jump_z",np.nan)>=SEAM_Z_THRESHOLD and
                abs(r.get("col_index",-999)-ar.get("col_index",-999))<=3
            ):
                match_rows.append({
                    "combination":ar["combination"],"model":ar["model"],
                    "axis":"column","annual_index":ar.get("col_index"),
                    "predictor":feat,"month":r["month"],
                    "predictor_index":r.get("col_index"),
                    "predictor_jump_z":r.get("max_col_jump_z")
                })

match_df=pd.DataFrame(match_rows)
match_df.to_csv(QA_DIR/"prediction_predictor_seam_matches_FIXED.csv",index=False)

if len(match_df):
    display(match_df.sort_values("predictor_jump_z",ascending=False).head(100))
else:
    print("No same-location strong predictor seam matches found.")

No same-location strong predictor seam matches found.


## 15. Final report

In [17]:
lines=[
    "FIXED SPATIAL DOWNSCALING - FINAL REPORT",
    "="*70,
    f"Project root: {PROJECT_ROOT}",
    f"Reference raster: {REFERENCE_RASTER}",
    f"Reference CRS: {REF['crs']}",
    f"Reference size: {REF['width']} x {REF['height']}",
    f"Reference resolution: {REF['res']}",
    "",
    "CORE FIXES APPLIED:",
    "- No nearest fill of internal NoData.",
    "- Rasterio reproject used instead of manual map_coordinates.",
    "- One explicit reference transform/grid used.",
    "- Continuous and categorical resampling separated.",
    "- Prediction only where all required predictors are valid.",
    "- Khulna boundary mask applied.",
    "- Annual maps are sums of 12 valid monthly predictions.",
    "",
    f"Predictor QA problems/flags: {len(problems)}",
    f"Annual maps generated: {int((annual_df['status']=='OK').sum()) if len(annual_df) else 0}",
    f"Annual seam flags: {int(annual_df['seam_flag'].fillna(False).sum()) if 'seam_flag' in annual_df else 0}",
    "",
    "IMPORTANT OUTPUTS:",
    str(QA_DIR/"predictor_QA_2022_PROBLEMS.csv"),
    str(QA_DIR/"monthly_prediction_QA.csv"),
    str(QA_DIR/"annual_seam_check_FIXED.csv"),
    str(QA_DIR/"prediction_predictor_seam_matches_FIXED.csv"),
    str(TABLE_DIR/"annual_output_statistics_FIXED.csv"),
    str(ANNUAL_DIR),
    str(FIG_DIR),
]

report=OUT_ROOT/"FINAL_FIXED_PIPELINE_REPORT.txt"
report.write_text("\\n".join(lines),encoding="utf-8")
print(report.read_text(encoding="utf-8"))

FIXED SPATIAL DOWNSCALING - FINAL REPORT\n======================================================================\nProject root: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\nReference raster: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\aligned_rasters\predictors\DEM\Khulna_SRTM_DEM.tif\nReference CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]\nReference size: 59 x 151\nReference resolution: (0.008983152841195215, 0.008983152841195215)\n\nCORE FIXES APPLIED:\n- No nearest fill of internal NoData.\n- Rasterio reproject used instead of manual map_coordinates.\n- One explicit reference transform/grid used.\n- Continuous and categorical

## After running, send me these 4 files

1. `FINAL_FIXED_PIPELINE_REPORT.txt`
2. `quality_control/predictor_QA_2022_PROBLEMS.csv`
3. `quality_control/annual_seam_check_FIXED.csv`
4. `quality_control/prediction_predictor_seam_matches_FIXED.csv`

Then the remaining source of the spatial variation can be identified precisely.